# GMM の k と共分散型を BIC で選ぶ

**出典**　機械学習 4 分野の教科書 — 第 9 章 3-A クラスタリング ／ 08 実装：4 手法を同じデータで比較する

**このノートのパス**　`ml-code/09_clustering/03_gmm_bic.ipynb`

**このノートで確かめること**
**BIC は小さいほど良い**——k と `covariance_type` を同時に選ぶ。
さらに **GMM は「新しい点の所属確率」を返せる**という、k-means にはできないことを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn numpy` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

In [3]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

RS = 42
X_blob, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.1, random_state=RS)
X_blob = StandardScaler().fit_transform(X_blob)

In [4]:
best = None
for cov in ["full", "tied", "diag", "spherical"]:
    for k in range(2, 9):
        gm = GaussianMixture(n_components=k, covariance_type=cov,
                             random_state=RS).fit(X_blob)
        bic = gm.bic(X_blob)                # 小さいほど良い
        if best is None or bic < best[0]:
            best = (bic, cov, k)

print(f"BIC 最小: covariance_type={best[1]}, k={best[2]}, BIC={best[0]:.1f}")

  File "<local-python>/lib/python3.10/site-packages/joblib/externals/loky/backend/context.py", line 264, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
  File "<local-python>/lib/python3.10/site-packages/joblib/externals/loky/backend/context.py", line 420, in _count_physical_cores_darwin
    return int(cpu_info)


BIC 最小: covariance_type=tied, k=3, BIC=556.8


In [5]:
# 学習済み GMM は「新しい点の所属確率」も返せる — k-means にはできないこと
gm = GaussianMixture(n_components=best[2], covariance_type=best[1],
                     random_state=RS).fit(X_blob)
proba = gm.predict_proba(X_blob[:5])
print(np.round(proba, 3))     # 各行の合計が 1 になる
print("各行の合計:", np.round(proba.sum(axis=1), 6))

[[0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
各行の合計: [1. 1. 1. 1. 1.]


### 境界付近の点ほど、所属が曖昧になる

In [6]:
p = gm.predict_proba(X_blob)
conf = p.max(axis=1)                       # 最も高い所属確率＝確信度
print(f"確信度 0.9 以上の点: {(conf >= 0.9).mean():.1%}")
print(f"確信度 0.6 未満の点: {(conf <  0.6).mean():.1%}  ← 境界付近")

order = np.argsort(conf)
print("\n最も曖昧な 3 点の所属確率:")
print(np.round(p[order[:3]], 3))

確信度 0.9 以上の点: 100.0%
確信度 0.6 未満の点: 0.0%  ← 境界付近

最も曖昧な 3 点の所属確率:
[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


## 読み取りどころ

- **BIC は「当てはまりの良さ」から「パラメータ数のペナルティ」を引いた指標**である。k を増やせば当てはまりは良くなるが、ペナルティも増えるので**最小値が存在する**。エルボー法のような主観が入らない。
- **`covariance_type` も一緒に選ぶ。**`full` は自由だがパラメータが多く、`spherical` は k-means に近い。**データが少ないときほど制約の強い型が選ばれやすい。**
- **`predict_proba` が返す「所属確率」は、k-means には無い出力**である。境界付近の点を「どちらとも言えない」として扱えるので、**1-D の LOW/REVIEW/HIGH のような 3 段設計**と相性がよい。

⚠️ BIC が選ぶのは**「このモデル族の中で最もつじつまが合う k」**であって、「真のクラスタ数」ではない。データがガウス混合から生成されていなければ、その保証はない。